In [1]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

100%|██████████| 26.4M/26.4M [00:32<00:00, 812kB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 109kB/s]
100%|██████████| 4.42M/4.42M [00:05<00:00, 765kB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.27MB/s]


In [2]:
num_inputs = 784
num_outputs = 10

W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs, requires_grad=True)

In [4]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition

In [5]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

In [6]:
def cross_loss(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y] )

In [8]:
def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

In [17]:
def evaluate_accuracy(net, data_iter):
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = Accumulator(2)
    for X, y in data_iter:
        metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

In [18]:
def train_epoch_ch3(net, train_iter, loss, updater):
    if isinstance(net, torch.nn.Module):
        net.train()
    metric = Accumulator(3)
    for X, y in train_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(
                float(l) * len(y),
                accuracy(y_hat, y),
                y.size().numel()
            )
        else:
            l.sum().backward()
            updater(X.shape[0])
            metric.add(
                float(l.sum()),
                accuracy(y_hat, y),
                y.numel()
            )
    return metric[0] / metric[2], metric[1] / metric[2]

In [25]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        train_loss, train_acc = train_metrics
        print(f'epoch {epoch + 1}, loss {train_loss:.3f}, 'f'train acc {train_acc:.3f}, test acc {test_acc:.3f}')

In [26]:
lr = 0.1

def updater(batch_size):
    return d2l.sgd([W, b], lr, batch_size)

In [27]:
num_epochs = 10
train_ch3(net, train_iter, test_iter, cross_loss, num_epochs, updater)

epoch 1, loss 0.443, train acc 0.849, test acc 0.832
epoch 2, loss 0.439, train acc 0.851, test acc 0.833
epoch 3, loss 0.436, train acc 0.852, test acc 0.836
epoch 4, loss 0.433, train acc 0.853, test acc 0.836
epoch 5, loss 0.431, train acc 0.854, test acc 0.836
epoch 6, loss 0.428, train acc 0.854, test acc 0.837
epoch 7, loss 0.425, train acc 0.855, test acc 0.837
epoch 8, loss 0.424, train acc 0.855, test acc 0.839
epoch 9, loss 0.422, train acc 0.856, test acc 0.839
epoch 10, loss 0.420, train acc 0.856, test acc 0.835


In [15]:
class Accumulator:
    """在 n 个变量上累加。"""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [
            a + float(b) for a, b in zip(self.data, args)
        ]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]